# 08 — Esgoto e separação em indireta e direta

Steps 2 and 3 of the chain, from the meter's hourly caudal (July–October 2026, including the new October data):

1. height → caudal (not here: this notebook starts from the meter's caudal)
2. caudal − esgoto → afluências indevidas, with esgoto from a table by block of hours and kind of day
3. afluências indevidas → indireta (the slow part, read on nights without recent rain) + direta (the rest)

The code is in `afluencias.py` and the checks in `verificacoes.py`. Every chart compares with the current method in `previsao.py` (one constant esgoto, one constant indireta, each piece clipped at zero).

Change `ESTACAO` and run all cells again.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
predict_dir = cwd if (cwd / "afluencias.py").exists() else cwd / "predict inflow"
for path in (predict_dir.parent, predict_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

import afluencias as a
import verificacoes as v
from pipeline_config import STATIONS

ESTACAO = "antanhol"  # ou "ameal"

flow = a.leituras(ESTACAO)
caudal = a.caudal_horario(flow)
chuva = a.chuva_horaria(ESTACAO, caudal)
novo = a.separar(caudal, chuva)
atual = a.separar_metodo_atual(flow, caudal, chuva)
secos_com_caudal = sorted(set(caudal.index.date) & novo.secos)
print(f"{STATIONS[ESTACAO]['folder']}: {len(caudal)} horas com caudal, {caudal.index.min():%d/%m} → {caudal.index.max():%d/%m}")
print(f"dias secos com caudal: {len(secos_com_caudal)}; madrugadas lidas para a indireta: {len(novo.noites)}")

Antanhol: 1311 horas com caudal, 22/07 → 14/10
dias secos com caudal: 32; madrugadas lidas para a indireta: 46


## The esgoto table

Typical esgoto for each 3-hour block, on working days and rest days (weekends and holidays), from the dry days. The dashed line is the single constant the current method uses. The numbers under the chart are how many dry days each value comes from.

In [2]:
t = novo.tabela
rotulos = [f"{b * a.HORAS_POR_BLOCO:02d}–{(b + 1) * a.HORAS_POR_BLOCO:02d}h" for b in t.valores.index]
fig = go.Figure()
fig.add_trace(go.Bar(x=rotulos, y=t.valores["util"], name="dias úteis", marker_color="#185FA5",
                     text=t.dias["util"], textposition="outside"))
fig.add_trace(go.Bar(x=rotulos, y=t.valores["descanso"], name="dias de descanso", marker_color="#D85A30",
                     text=t.dias["descanso"], textposition="outside"))
fig.add_hline(y=float(atual["esgoto"].max()), line=dict(color="gray", dash="dash"),
              annotation_text=f"método atual: {atual['esgoto'].max():.2f} m³/h")
fig.update_layout(barmode="group", height=420, template="plotly_white", yaxis_title="esgoto (m³/h)",
                  title="Esgoto típico por bloco de horas (número = dias secos usados)")
fig.show()

## The whole period, hour by hour

Top: rain. Middle: the meter's caudal, the typical esgoto from the table, and the indireta line (dots: the nights where we read it). Bottom: afluências indevidas split into indireta and direta. Direta can go below zero in an hour when sewage is below its typical value; that is expected, and it cancels out over a day.

In [3]:
h = novo.horas.reindex(pd.date_range(caudal.index.min(), caudal.index.max(), freq="h"))
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.15, 0.45, 0.4],
                    subplot_titles=("Chuva (mm/h)", "Caudal, esgoto típico e indireta (m³/h)", "Afluências indevidas = indireta + direta (m³/h)"))
c_ = chuva.loc[caudal.index.min():caudal.index.max()]
fig.add_trace(go.Bar(x=c_.index, y=c_, marker_color="#378ADD", name="chuva"), row=1, col=1)
fig.add_trace(go.Scatter(x=h.index, y=h["caudal"], name="caudal (medidor)", line=dict(color="#26215C", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=h.index, y=h["esgoto"] + h["indireta"], name="esgoto típico + indireta", line=dict(color="#888780", width=1, dash="dot")), row=2, col=1)
fig.add_trace(go.Scatter(x=h.index, y=h["indireta"], name="indireta", line=dict(color="#0F6E56", width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=novo.noites.index, y=novo.noites, mode="markers", name="madrugadas lidas", marker=dict(color="#0F6E56", size=6)), row=2, col=1)
fig.add_trace(go.Scatter(x=h.index, y=h["indireta"], name="indireta", line=dict(color="#0F6E56", width=0), fill="tozeroy", fillcolor="rgba(15,110,86,0.35)", showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=h.index, y=h["afluencias_indevidas"], name="afluências indevidas", line=dict(color="#D85A30", width=1)), row=3, col=1)
fig.update_layout(height=900, template="plotly_white", hovermode="x unified")
fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.04), row=3, col=1)
fig.show()

## Check 1: direta on dry days, measured fairly

On a dry day, direta should be close to zero. Because the table is made from the dry days, we leave each dry day out of the table and measure its direta. Each bar is one dry day, as a percentage of that day's caudal. The current method is shown for comparison (it does not need a table, so no day is left out).

In [4]:
fora = v.direta_dias_secos_fora(caudal, chuva)
caudal_dia = caudal.groupby(caudal.index.date).mean()
atual_dia = atual["direta"].groupby(atual.index.date).mean()
dias = [pd.Timestamp(d) for d in fora.index]
fig = go.Figure()
fig.add_trace(go.Bar(x=dias, y=100 * fora / caudal_dia.reindex(fora.index), name="método novo", marker_color="#185FA5"))
fig.add_trace(go.Bar(x=dias, y=100 * atual_dia.reindex(fora.index) / caudal_dia.reindex(fora.index), name="método atual", marker_color="#B4B2A9"))
fig.update_layout(barmode="group", height=380, template="plotly_white", yaxis_title="direta média do dia (% do caudal)",
                  title="Direta nos dias secos (deveria ser perto de 0)")
fig.show()
print(f"método novo: média {100 * fora.mean() / caudal_dia.reindex(fora.index).mean():.1f} %, "
      f"um dia típico afasta-se {100 * fora.abs().mean() / caudal_dia.reindex(fora.index).mean():.0f} %")
print(f"método atual: média {100 * atual_dia.reindex(fora.index).mean() / caudal_dia.reindex(fora.index).mean():.1f} %")

método novo: média 1.0 %, um dia típico afasta-se 19 %
método atual: média 16.9 %


## Check 2: daily totals

Afluências indevidas per day should not be negative. Days below zero are days when the caudal was below the typical esgoto: either sewage really was lower that day (holidays, people away), or something else in the network changed the flow.

In [5]:
diarios = v.totais_diarios(novo.horas)
dias = [pd.Timestamp(d) for d in diarios.index]
fig = go.Figure()
fig.add_trace(go.Bar(x=dias, y=diarios["indireta"], name="indireta", marker_color="#0F6E56"))
fig.add_trace(go.Bar(x=dias, y=diarios["direta"], name="direta", marker_color="#D85A30"))
fig.add_trace(go.Scatter(x=dias, y=diarios["afluencias_indevidas"], name="afluências indevidas", mode="markers", marker=dict(color="black", size=6)))
fig.update_layout(barmode="relative", height=420, template="plotly_white", yaxis_title="m³/dia",
                  title="Afluências indevidas por dia (método novo)")
fig.show()
display(pd.DataFrame({"método novo": v.resumo(novo.horas), "método atual": v.resumo(atual)}).round(1))

,método novo,método atual
dias com afluências indevidas negativas,15.0,0.0
dias com totais,53.0,53.0
afluências indevidas no período (m³),8889.4,6839.2
indireta no período (m³),8964.7,1630.3
direta no período (m³),-75.3,5208.9


## Check 3: real storms

Direta for each real storm of 2 mm or more, per mm of rain. Read as an area: if all the rain on an area A entered the sewer, 1 mm would give A/1000 m³. So the "connected area" is the area of roofs and drains that would have to be connected to the sewer to give that direta. It should be similar from storm to storm, and plausible for the area the pipe serves. `horas_medidas` is the share of the storm window that had caudal. Storms with caudal in less than half the window are left out.

In [6]:
display(pd.concat({"método novo": v.direta_por_tempestade(novo.horas, chuva).set_index("inicio"),
                   "método atual": v.direta_por_tempestade(atual, chuva).set_index("inicio")}, axis=1).round(2))

método novo                                           \
                       chuva_mm horas_medidas direta_m3 direta_m3_por_mm   
inicio                                                                     
2026-07-25 10:00:00         2.3          0.89    294.21           127.92   
2026-08-23 17:00:00        26.9          1.00    194.33             7.22   
2026-08-25 20:00:00        28.2          1.00    510.17            18.09   
2026-10-05 13:00:00         7.0          0.90   -497.31           -71.04   

                                   método atual                          \
                    area_ligada_m2     chuva_mm horas_medidas direta_m3   
inicio                                                                    
2026-07-25 10:00:00      127917.94          2.3          0.89    439.89   
2026-08-23 17:00:00        7224.03         26.9          1.00    143.45   
2026-08-25 20:00:00       18091.04         28.2          1.00    489.75   
2026-10-05 13:00:00      -71043.74          7.0          0.90      0.65   

                                                     
                    direta_m3_por_mm area_ligada_m2  
inicio                                               
2026-07-25 10:00:00           191.25      191254.38  
2026-08-23 17:00:00             5.33        5332.61  
2026-08-25 20:00:00            17.37       17366.92  
2026-10-05 13:00:00             0.09          93.46

## Check 4: made-up storms

We add storms of our own to the meter's real caudal on dry days: a fast direta spike and a slow indireta rise, with volumes we choose, plus the matching rain. Then we run both methods and see what share of the added volumes they find (100 % is perfect). The same rain goes into the runs with and without the added water, so the only difference between them is the water. This tests the method, not whether real storms behave like the made-up ones.

In [7]:
try:
    inventadas = v.tempestades_inventadas(flow, chuva, n=30)
    display(v.recuperacao(inventadas).round(0))
except ValueError as erro:
    inventadas = None
    print("Não é possível fazer este teste nesta estação:", erro)

,método,parte,"encontrado (mediana, %)",10 % das vezes abaixo de,10 % das vezes acima de
0,método novo,direta,100.0,96.0,103.0
1,método novo,indireta,102.0,97.0,107.0
2,método novo,afluencias_indevidas,100.0,100.0,101.0
3,método atual,direta,86.0,47.0,130.0
4,método atual,indireta,2.0,0.0,469.0
5,método atual,afluencias_indevidas,58.0,34.0,290.0


In [8]:
# Uma das tempestades inventadas, desenhada: o que acrescentámos e o que o método novo encontrou.
if inventadas is not None:
    exemplo = v.Tempestade(inicio=inventadas.loc[0, "inicio"], chuva_mm=inventadas.loc[0, "chuva_mm"],
                           volume_direta=caudal.mean() * 24 * 0.3, volume_indireta=caudal.mean() * 24 * 0.3)
    novo_flow, nova_chuva, direta_add, indireta_add = v.com_tempestade(flow, chuva, exemplo)
    antes = a.separar(caudal, nova_chuva).horas
    depois = a.separar(a.caudal_horario(novo_flow), nova_chuva).horas
    janela = slice(exemplo.inicio - pd.Timedelta(days=1), exemplo.inicio + pd.Timedelta(days=6))
    d, i = (depois - antes).loc[janela], antes.loc[janela]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=("Direta: acrescentada vs encontrada (m³/h)", "Indireta: acrescentada vs encontrada (m³/h)"))
    fig.add_trace(go.Scatter(x=direta_add.loc[janela].index, y=direta_add.loc[janela], name="direta acrescentada", line=dict(color="black", dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=d["direta"], name="direta encontrada", line=dict(color="#D85A30")), row=1, col=1)
    fig.add_trace(go.Scatter(x=indireta_add.loc[janela].index, y=indireta_add.loc[janela], name="indireta acrescentada", line=dict(color="black", dash="dash")), row=2, col=1)
    fig.add_trace(go.Scatter(x=d.index, y=d["indireta"], name="indireta encontrada", line=dict(color="#0F6E56")), row=2, col=1)
    fig.update_layout(height=600, template="plotly_white", hovermode="x unified")
    fig.show()